# 06 -- Retrieval Evaluation (P@10)
Compares BM25, Dense, and Hybrid+Rerank 


In [1]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/multimodal_rag"))

os.environ["JAVA_HOME"] = "/home/aakul001/.conda/envs/rag310/lib/jvm"
os.environ["JVM_PATH"]  = "/home/aakul001/.conda/envs/rag310/lib/jvm/lib/server/libjvm.so"

from configs import config
from src.utils import setup_java
setup_java()
print("Setup done")

[21:02:52] INFO utils -- JAVA_HOME=/home/aakul001/.conda/envs/rag310/lib/jvm


Setup done


## Load all models

In [2]:

from src.retrieval import (load_bm25, load_text_model, load_reranker,
                            load_faiss_text, load_faiss_image, load_metadata)

load_bm25()
load_text_model()
load_reranker()
load_faiss_text()
load_faiss_image()
load_metadata()
print("All models loaded")


/home/aakul001/.conda/envs/rag310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[21:03:39] INFO retrieval -- Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index

2026-04-26 21:03:39,943 - retrieval - INFO - Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index
Apr 26, 2026 9:03:39 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
[21:03:40] INFO retrieval -- BM25 ready

2026-04-26 21:03:40,427 - retrieval - INFO - BM25 ready
[21:03:42] INFO retrieval -- Loading text embedding model: FremyCompany/BioLORD-2023-C

2026-04-26 21:03:42,444 - retrieval - INFO - Loading text embedding model: FremyCompany/BioLORD-2023-C
Loadi

All models loaded


## load queries

In [3]:
from src.utils import load_jsonl
from configs.config import QUERIES_FILE

queries = load_jsonl(QUERIES_FILE)
for q in queries:
    print(f"  [{q['qid']}] {q['query']}")

  [q01] COVID-19 impact on healthcare systems
  [q02] digital health transformation in primary care
  [q03] gut microbiota changes during viral infection
  [q04] chest X-ray findings in pneumonia
  [q05] hypertension treatment in primary care settings
  [q06] what are the benefits of telemedicine for rural patients
  [q07] MRI brain scan interpretation in neurological disorders
  [q08] how does COVID-19 affect lung tissue
  [q09] antibiotic resistance mechanisms in bacteria
  [q10] nurse leadership in digital healthcare innovation


In [5]:
#load_image_model in retrieval.py to use open_clip
path = "/home/aakul001/multimodal_rag/src/retrieval.py"
content = open(path).read()

old = '''def load_image_model():
    global _image_model, _image_processor
    if _image_model is not None:
        return _image_model, _image_processor
    from configs.config import IMAGE_EMBED_MODEL
    from transformers import AutoProcessor, AutoModel
    logger.info(f"Loading image model: {IMAGE_EMBED_MODEL}")
    _image_processor = AutoProcessor.from_pretrained(IMAGE_EMBED_MODEL)
    _image_model     = AutoModel.from_pretrained(IMAGE_EMBED_MODEL)
    _image_model.eval()
    logger.info("Image model ready")
    return _image_model, _image_processor'''

new = '''def load_image_model():
    global _image_model, _image_processor
    if _image_model is not None:
        return _image_model, _image_processor
    import open_clip
    import torch
    logger.info("Loading BiomedCLIP via open_clip...")
    model, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    _image_model     = model
    _image_processor = preprocess
    logger.info(f"BiomedCLIP ready on {device}")
    return _image_model, _image_processor'''

if old in content:
    content = content.replace(old, new)
    open(path, "w").write(content)
    print("Fixed successfully")
else:
    print("Pattern not found - printing current function for inspection:")
    start = content.find("def load_image_model")
    print(content[start:start+600])

Fixed successfully


In [6]:
# search_dense_image_from_query to use open_clip encode_text
path = "/home/aakul001/multimodal_rag/src/retrieval.py"
content = open(path).read()

old = '''    with torch.no_grad():
        inputs    = processor(text=[query], return_tensors="pt", padding=True, truncation=True)
        text_feat = model.get_text_features(**inputs)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)'''

new = '''    import open_clip
    tokenizer = open_clip.get_tokenizer("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    with torch.no_grad():
        text_tokens = tokenizer([query]).to(next(model.parameters()).device)
        text_feat   = model.encode_text(text_tokens)
        text_feat   = text_feat / text_feat.norm(dim=-1, keepdim=True)'''

if old in content:
    content = content.replace(old, new)
    open(path, "w").write(content)
    print("Fixed successfully")
else:
    print("Pattern not found")

Fixed successfully


In [7]:
import importlib, src.retrieval
importlib.reload(src.retrieval)
from src.retrieval import (load_bm25, load_text_model, load_reranker,
                            load_faiss_text, load_faiss_image, load_metadata,
                            retrieve_all_systems)

load_bm25()
load_text_model()
load_reranker()
load_faiss_text()
load_faiss_image()
load_metadata()
print("All models reloaded")

[21:12:37] INFO retrieval -- Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index

2026-04-26 21:12:37,237 - retrieval - INFO - Loading BM25 index: /home/aakul001/multimodal_rag/indexes/bm25_index
[21:12:37] INFO retrieval -- BM25 ready

2026-04-26 21:12:37,266 - retrieval - INFO - BM25 ready
[21:12:37] INFO retrieval -- Loading text embedding model: FremyCompany/BioLORD-2023-C

2026-04-26 21:12:37,268 - retrieval - INFO - Loading text embedding model: FremyCompany/BioLORD-2023-C
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3874.83it/s]
[21:12:38] INFO retrieval -- Text model ready

2026-04-26 21:12:38,271 - retrieval - INFO - Text model ready
[21:12:38] INFO retrieval -- Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

2026-04-26 21:12:38,273 - retrieval - INFO - Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4470.95it/s]
[21:12:39] INFO retrieval -- Reranker ready

2026-04-26 21:12:39,53

All models reloaded


## run all three systems

In [8]:
import pandas as pd
from src.retrieval import retrieve_all_systems
from configs.config import RESULTS_DIR, FINAL_TOP_K

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for q in queries:
    qid, query = q["qid"], q["query"]
    print(f"Running {qid}...")
    systems = retrieve_all_systems(query, top_k=FINAL_TOP_K)
    for system, df in systems.items():
        df = df.copy()
        df["query"]  = query
        df["qid"]    = qid
        df["system"] = system
        all_results.append(df)

results_df = pd.concat(all_results, ignore_index=True)
out = RESULTS_DIR / "retrieval_results.csv"
results_df.to_csv(out, index=False)
print(f"Saved: {out}")
print(results_df[["qid","system","rank","modality"]].head(15))

Running q01...


[21:13:12] INFO retrieval -- Loading BiomedCLIP via open_clip...

2026-04-26 21:13:12,166 - retrieval - INFO - Loading BiomedCLIP via open_clip...
[21:13:17] INFO retrieval -- BiomedCLIP ready on cuda

2026-04-26 21:13:17,267 - retrieval - INFO - BiomedCLIP ready on cuda


Running q02...
Running q03...
Running q04...
Running q05...
Running q06...
Running q07...
Running q08...
Running q09...
Running q10...
Saved: /home/aakul001/multimodal_rag/results/retrieval_results.csv
    qid system  rank modality
0   q01   BM25     1     text
1   q01   BM25     2     text
2   q01   BM25     3     text
3   q01   BM25     4     text
4   q01   BM25     5     text
5   q01   BM25     6     text
6   q01   BM25     7     text
7   q01   BM25     8     text
8   q01   BM25     9     text
9   q01   BM25    10     text
10  q01  Dense     1     text
11  q01  Dense     2     text
12  q01  Dense     3     text
13  q01  Dense     4     text
14  q01  Dense     5     text


## Judgment template

Fill in the `relevant` column (1=relevant, 0=not).

In [9]:
from configs.config import JUDGMENTS_FILE, EVAL_DIR
from src.evaluation import create_judgment_template

EVAL_DIR.mkdir(parents=True, exist_ok=True)

if not JUDGMENTS_FILE.exists():
    create_judgment_template(results_df, JUDGMENTS_FILE)
    print(f"Created: {JUDGMENTS_FILE}")
    print("NOW open this file and fill the relevant column")
    print("1 = passage answers the query, 0 = does not")
else:
    print(f"Already exists: {JUDGMENTS_FILE}")

[21:15:51] INFO evaluation -- Judgment template saved: /home/aakul001/multimodal_rag/evaluation/relevance_judgments.csv (218 pairs)

2026-04-26 21:15:51,229 - evaluation - INFO - Judgment template saved: /home/aakul001/multimodal_rag/evaluation/relevance_judgments.csv (218 pairs)


Created: /home/aakul001/multimodal_rag/evaluation/relevance_judgments.csv
NOW open this file and fill the relevant column
1 = passage answers the query, 0 = does not


## load results

In [20]:
import pandas as pd
from configs.config import RESULTS_DIR

results_df = pd.read_csv(RESULTS_DIR / "retrieval_results.csv")
print(f"Loaded {len(results_df)} rows")
print(results_df["system"].unique())

Loaded 300 rows
['BM25' 'Dense' 'Hybrid+Rerank']


## Compute P@10, NDCG@10, MRR

In [21]:
from src.evaluation import load_judgments, compute_metrics, print_report
from configs.config import JUDGMENTS_FILE, EVAL_TOP_K, RESULTS_DIR

judgments_df = load_judgments(JUDGMENTS_FILE)
metrics_df   = compute_metrics(results_df, judgments_df, k=EVAL_TOP_K)
print_report(metrics_df, k=EVAL_TOP_K)
metrics_df.to_csv(RESULTS_DIR / "metrics.csv", index=False)
print("Metrics saved")

[22:05:52] INFO evaluation -- Loaded 218 judgments: 179 relevant, 39 not relevant

2026-04-26 22:05:52,391 - evaluation - INFO - Loaded 218 judgments: 179 relevant, 39 not relevant



  Retrieval Evaluation -- P@10 / NDCG@10 / MRR
               P@10  NDCG@10     MRR
system                              
Hybrid+Rerank  0.93   0.9918  1.0000
Dense          0.89   0.9415  0.8833
BM25           0.74   0.9022  0.9000

Per-query breakdown:

  [BM25]
    COVID-19 impact on healthcare systems               0.90  |##################
    MRI brain scan interpretation in neurological diso  0.70  |##############
    antibiotic resistance mechanisms in bacteria        1.00  |####################
    chest X-ray findings in pneumonia                   0.40  |########
    digital health transformation in primary care       0.60  |############
    gut microbiota changes during viral infection       0.80  |################
    how does COVID-19 affect lung tissue                0.80  |################
    hypertension treatment in primary care settings     0.70  |##############
    nurse leadership in digital healthcare innovation   0.80  |################
    what are the benefits